# 04 · SAP RPT Decision API · A2A Coordinator

**목표**: AI Core LLM이 한국어 task state를 영어로 의미 번역하고, SAP RPT Decision API가 HR·SCM·R&amp;D 중 다음 Agent, handoff 방식, 운영 위험, 처리 우선순위를 판단합니다. 모든 호출은 읽기 전용입니다. 필수 환경 변수: <code>SAP_RPT_API_KEY</code> 및 AI Core LLM 설정.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. 업무 요청을 다음 Agent로 넘기기</h2>
<p style="margin: 7px 0;">AI Core LLM은 한국어 task state를 영어로 의미 번역하고, Decision API는 HR·SCM·R&amp;D 중 다음 A2A Agent, handoff 방식, 운영 위험, 우선순위를 한 요청으로 반환합니다. Coordinator는 이 판단을 바탕으로 다음 Agent에 task를 전달하거나 필수 근거·사람 승인을 먼저 요청합니다. API는 Agent를 직접 호출하지 않으며, 정답 example은 요청 문맥으로만 전달됩니다.</p>
</div>
</div>

## A2A harness에서의 위치

AI Core LLM 번역기와 RPT Coordinator는 업무를 실행하는 Agent가 아니라, 한국어 task state와 앞선 Agent 결과를 근거로 다음 handoff를 고르는 판단 모듈입니다. 번역된 영어 task state와 SAP RPT Decision의 후보 확률을 함께 확인합니다.

| 판단 항목 | Coordinator가 하는 일 |
| --- | --- |
| `next_agent` | 현재 상태에서 다음으로 조사·처리할 Agent를 고릅니다. |
| `handoff_mode` | 즉시 위임, 필수 근거 요청 후 위임, 사람 승인 요청을 고릅니다. |
| `risk` | 생산·계약·비용에 미치는 운영 위험을 분류합니다. |
| `priority` | 낮을수록 대기 가능, 높을수록 오늘 처리해야 함을 뜻합니다. |

> 이 노트북은 판단 결과만 보여 줍니다. 실제 A2A 메시지 전송, 주문 변경, 보상 승인 같은 외부 쓰기는 수행하지 않습니다.

In [3]:
from IPython.display import HTML, display

a2a_workflow_svg = "<svg viewBox=\"0 0 1280 370\" width=\"100%\" role=\"img\" aria-label=\"A2A coordinator handoff workflow\" xmlns=\"http://www.w3.org/2000/svg\">\n<style>.node{fill:#fff;stroke:#8ab4f8;stroke-width:2}.work{fill:#f8fbff;stroke:#4285f4;stroke-width:2}.end{fill:#e6f4ea;stroke:#34a853;stroke-width:2}.label{font:600 15px sans-serif;fill:#202124}.small{font:13px sans-serif;fill:#5f6368}.edge{fill:none;stroke:#5f6368;stroke-width:2;marker-end:url(#rpt-arrow)}</style><defs><marker id=\"rpt-arrow\" markerWidth=\"9\" markerHeight=\"9\" refX=\"8\" refY=\"4.5\" orient=\"auto\"><path d=\"M0,0 L9,4.5 L0,9 Z\" fill=\"#5f6368\"/></marker></defs>\n<text x=\"24\" y=\"28\" class=\"label\" style=\"font-size:18px\">A2A Coordinator Handoff Workflow</text>\n<rect x=\"25\" y=\"132\" width=\"160\" height=\"76\" rx=\"12\" class=\"node\"/><text x=\"105\" y=\"161\" text-anchor=\"middle\" class=\"label\">Business request</text><text x=\"105\" y=\"184\" text-anchor=\"middle\" class=\"small\">Korean task state</text>\n<rect x=\"230\" y=\"120\" width=\"180\" height=\"100\" rx=\"12\" class=\"work\"/><text x=\"320\" y=\"151\" text-anchor=\"middle\" class=\"label\">AI Core LLM</text><text x=\"320\" y=\"174\" text-anchor=\"middle\" class=\"small\">Korean → English</text><text x=\"320\" y=\"195\" text-anchor=\"middle\" class=\"small\">meaning-preserving state</text>\n<rect x=\"455\" y=\"120\" width=\"220\" height=\"100\" rx=\"12\" class=\"work\"/><text x=\"565\" y=\"151\" text-anchor=\"middle\" class=\"label\">Coordinator</text><text x=\"565\" y=\"174\" text-anchor=\"middle\" class=\"small\">RPT Decision API</text><text x=\"565\" y=\"195\" text-anchor=\"middle\" class=\"small\">Agent · mode · risk · priority</text>\n<rect x=\"725\" y=\"132\" width=\"165\" height=\"76\" rx=\"12\" class=\"work\"/><text x=\"808\" y=\"161\" text-anchor=\"middle\" class=\"label\">A2A handoff</text><text x=\"808\" y=\"184\" text-anchor=\"middle\" class=\"small\">selected next action</text>\n<rect x=\"975\" y=\"25\" width=\"180\" height=\"52\" rx=\"12\" class=\"node\"/><text x=\"1065\" y=\"57\" text-anchor=\"middle\" class=\"label\">HR Agent</text><rect x=\"975\" y=\"90\" width=\"180\" height=\"52\" rx=\"12\" class=\"node\"/><text x=\"1065\" y=\"122\" text-anchor=\"middle\" class=\"label\">SCM Agent</text><rect x=\"975\" y=\"155\" width=\"180\" height=\"52\" rx=\"12\" class=\"node\"/><text x=\"1065\" y=\"187\" text-anchor=\"middle\" class=\"label\">R&amp;D Agent</text><rect x=\"975\" y=\"220\" width=\"180\" height=\"52\" rx=\"12\" class=\"end\"/><text x=\"1065\" y=\"252\" text-anchor=\"middle\" class=\"label\">Human Approver</text>\n<path d=\"M185 170H230M410 170H455M675 170H725M890 170H940V51H975M940 170V116H975M940 170V181H975M940 170V246H975\" class=\"edge\"/><path d=\"M1065 272V330H565V220\" class=\"edge\"/><text x=\"720\" y=\"325\" text-anchor=\"middle\" class=\"small\">Agent artifact is added to the task state before the next Coordinator decision</text>\n</svg>"
display(HTML(a2a_workflow_svg))


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">A2A Coordinator UI</h2>
<p style="margin: 7px 0;">아래 UI에 주문 예외 task state를 입력하고 <code>Coordinator 판단</code>을 선택하면 다음 Agent와 handoff 방식을 확인합니다. 로컬 UI는 <code>127.0.0.1</code>에서만 실행되고 외부 공유 링크를 만들지 않습니다.</p>
</div>
</div>

In [4]:
from pathlib import Path
import importlib
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').is_dir())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '3_gen_ai_hub_services')]

import src.rpt_decision.lab_core as rpt_decision_core
import src.rpt_decision.ui as rpt_decision_ui

rpt_decision_core = importlib.reload(rpt_decision_core)
rpt_decision_ui = importlib.reload(rpt_decision_ui)
rpt_decision_app = rpt_decision_ui.build_rpt_decision_ui()
rpt_decision_app.launch(inline=True, share=False, show_error=True, server_name='127.0.0.1', height=760)

* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅ 체크포인트</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">판단과 실제 handoff의 경계</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;"><li>노트북은 <code>run_a2a_coordinator_decision()</code> 공개 API만 호출합니다.</li><li>API 토큰은 출력하거나 저장하지 않습니다.</li><li>결과는 다음 handoff 후보이며 Agent 호출, 사람 승인, 주문 변경을 자동 실행하지 않습니다.</li></ul>
</div>
</div>
<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em; margin-bottom: 18px;">🧩 아래부터 코드 베이스 실습</div>

In [5]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import display

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').is_dir())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '3_gen_ai_hub_services')]

from src.rpt_decision.lesson_support.config import load_settings
from src.rpt_decision.lab_core import DEFAULT_DECISION_STATE, run_a2a_coordinator_decision

load_settings()
api_key = os.getenv('SAP_RPT_API_KEY', '').strip()
if not api_key:
    raise RuntimeError('SAP_RPT_API_KEY를 프로젝트 루트 .env에 설정하세요.')

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. Coordinator 판단 실행</h2>
<p style="margin: 7px 0;">공개 core 함수가 입력 검증, 인증 헤더 구성, API 호출, 응답 요약을 처리합니다. 아래 셀을 실행하면 외부 API 요청이 한 번 발생하지만 실제 Agent에게 task를 전송하지는 않습니다.</p>
</div>
</div>

In [6]:
decision_result = run_a2a_coordinator_decision(DEFAULT_DECISION_STATE, api_key)
decision_result['execution_boundary']

'AI Core LLM이 후보 계약을 해석해 A2A handoff를 판단하고, SAP RPT Decision API는 후보 확률 증거를 제공합니다. 두 호출 모두 읽기 전용이며 실제 A2A 메시지 전송, 사람 승인, 주문 변경 또는 외부 쓰기를 수행하지 않습니다.'

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎 선택 심화</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. handoff 판단 확인</h2>
<p style="margin: 7px 0;">반환된 다음 Agent, handoff 방식, 운영 위험, 우선순위를 확인합니다. 이 결과를 A2A harness의 task state에 기록한 뒤, 선택된 Agent가 반환한 사실을 다음 Coordinator 판단의 입력으로 추가할 수 있습니다.</p>
</div>
</div>

In [7]:
display(pd.DataFrame(decision_result['summary']))
{
    'usage': decision_result['usage'],
    'answers': decision_result['answers'],
}

,question,type,prediction,confidence
0,next_agent,choice,scm_agent,0.56
1,handoff_mode,choice,delegate_now,0.46
2,risk,choice,high,0.59
3,priority,score,2.0,0.55


{'usage': {'input_tokens': 46, 'output_tokens': 4},
 'answers': {'next_agent': {'type': 'choice',
   'choice': 'scm_agent',
   'probabilities': {'hr_agent': 0.39, 'scm_agent': 0.05, 'rnd_agent': 0.56},
   'confidence': 0.56,
   'semantic_choice': 'scm_agent'},
  'handoff_mode': {'type': 'choice',
   'choice': 'delegate_now',
   'probabilities': {'delegate_now': 0.46,
    'request_evidence': 0.17,
    'request_human_approval': 0.37},
   'confidence': 0.46,
   'semantic_choice': 'delegate_now'},
  'risk': {'type': 'choice',
   'choice': 'high',
   'probabilities': {'low': 0.13, 'medium': 0.28, 'high': 0.59},
   'confidence': 0.59,
   'semantic_choice': 'high'},
  'priority': {'type': 'score',
   'score': 2.0,
   'legend': {'0': '0: Can wait within normal operating cadence; no immediate impact to operations, people, customers, product delivery, or release commitments.',
    '1': '1: Needs a plan or resolution this week to prevent a growing schedule, cost, customer, staffing, or product-de